# RNA Preprocessing — METABRIC
Mirrors the logic of `02_rna_preprocessing.ipynb` (TCGA-BRCA).

Input: `METABRICK/preprocessed/metabric_rna_cleaned.csv` + `outcome.csv`

Output: 8 dataset variants saved to `METABRICK/v2/rna/statistical_filtered/`

| V | Name | Selection logic |
|---|---|---|
| 1 | ultra_conservative | Variance > 98.3 percentile |
| 2 | conservative | Variance > 97.5 percentile |
| 3 | standard | Variance > 95.9 percentile |
| 4 | fdr_significant | Spearman FDR < 0.05, top 1000 by composite |
| 5 | balanced | Var top 25% → Spearman top 10% |
| 6 | correlation | Abs. Spearman > 97.5 percentile |
| 7 | top_correlated | Top 100 positive + top 100 negative Spearman |
| 8 | composite | Average rank of Spearman + MI + Distance corr |

## 1. Paths

In [3]:
from pathlib import Path

# Walk up from notebook location until we find the METABRICK folder
_cwd = Path().resolve()
BASE_DIR = next(
    p for p in [_cwd, *_cwd.parents]
    if (p / "preprocessed" / "metabric_rna_cleaned.csv").exists()
)

RNA_FILE     = BASE_DIR / "preprocessed" / "metabric_rna_cleaned.csv"
OUTCOME_FILE = BASE_DIR / "preprocessed" / "outcome.csv"
OUTPUT_DIR   = BASE_DIR / "v2" / "rna" / "statistical_filtered"
STATS_CACHE  = BASE_DIR / "v2" / "rna" / "rna_statistics_cache.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_GENES    = 50
TARGET_BROAD = 1000   # target size for V4 and V8

assert RNA_FILE.exists(),     f"Not found: {RNA_FILE}"
assert OUTCOME_FILE.exists(), f"Not found: {OUTCOME_FILE}"

print(f"Base    : {BASE_DIR}")
print(f"RNA     : {RNA_FILE}")
print(f"Outcome : {OUTCOME_FILE}")
print(f"Output  : {OUTPUT_DIR}")
print(f"Cache   : {STATS_CACHE}")
print("Paths OK")

Base    : C:\Users\olegk\Desktop\Thesis_v3\03_METABRIC_external_validation
RNA     : C:\Users\olegk\Desktop\Thesis_v3\03_METABRIC_external_validation\preprocessed\metabric_rna_cleaned.csv
Outcome : C:\Users\olegk\Desktop\Thesis_v3\03_METABRIC_external_validation\preprocessed\outcome.csv
Output  : C:\Users\olegk\Desktop\Thesis_v3\03_METABRIC_external_validation\v2\rna\statistical_filtered
Cache   : C:\Users\olegk\Desktop\Thesis_v3\03_METABRIC_external_validation\v2\rna\rna_statistics_cache.csv
Paths OK


## 2. Imports

In [5]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, rankdata
from statsmodels.stats.multitest import multipletests
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")

print("Imports OK")

Imports OK


## 3. Load data

In [7]:
print("Loading RNA data...")
# metabric_rna_cleaned.csv is already samples x genes (rows = patients)
rna_raw = pd.read_csv(RNA_FILE, index_col=0)
print(f"  Raw shape  : {rna_raw.shape}  (samples x genes)")

print("\nLoading outcome...")
outcome = pd.read_csv(OUTCOME_FILE, index_col=0)
print(f"  Samples    : {len(outcome)}")
print(f"  Columns    : {outcome.columns.tolist()}")
print(f"  OS.time    : {outcome['OS.time'].min():.0f} – {outcome['OS.time'].max():.0f} days")
print(f"  Events     : {int(outcome['OS'].sum())} ({outcome['OS'].mean()*100:.1f}%)")

# Align on common samples
common  = sorted(set(rna_raw.index) & set(outcome.index))
rna_raw = rna_raw.loc[common].copy()
outcome = outcome.loc[common].copy()
print(f"  Overlap    : {len(common)} samples")

Loading RNA data...
  Raw shape  : (1980, 20385)  (samples x genes)

Loading outcome...
  Samples    : 1980
  Columns    : ['OS.time', 'OS']
  OS.time    : 0 – 10812 days
  Events     : 1143 (57.7%)
  Overlap    : 1980 samples


## 4. Quality control

In [9]:
print("Quality control...")

# Remove genes with >20% missing values
miss_frac = rna_raw.isna().mean(axis=0)
rna_qc = rna_raw.loc[:, miss_frac <= 0.20]
print(f"  After missing filter (>20%)  : {rna_qc.shape[1]:,} genes  "
      f"(dropped {rna_raw.shape[1] - rna_qc.shape[1]:,})")

# Impute residual missing values with per-gene median
imp = SimpleImputer(strategy="median")
rna_qc = pd.DataFrame(
    imp.fit_transform(rna_qc),
    index=rna_qc.index,
    columns=rna_qc.columns
)

# Remove zero-variance genes
var0 = rna_qc.var(axis=0)
rna_qc = rna_qc.loc[:, var0 > 0]
print(f"  After zero-variance removal  : {rna_qc.shape[1]:,} genes")

# Check value range to decide if log2 normalization is needed
_max = rna_qc.values.max()
_min = rna_qc.values.min()
print(f"\n  Value range: {_min:.2f} – {_max:.2f}")
print(f"  (If values look like raw counts >1000, log2 is applied below)")

Quality control...
  After missing filter (>20%)  : 20,385 genes  (dropped 0)
  After zero-variance removal  : 20,385 genes

  Value range: 4.56 – 14.88
  (If values look like raw counts >1000, log2 is applied below)


## 5. Normalization — log2(x+1) then z-score

> **Note:** `metabric_rna_cleaned.csv` is typically already log-transformed, but values
> can vary across versions. The cell below checks and applies log2 only if the data
> looks like it hasn't been transformed yet (max > 100 is a safe heuristic).

In [11]:
# Apply log2(x+1) only if data looks like raw/un-logged expression
if rna_qc.values.max() > 100:
    print("Applying log2(x+1) transform (values > 100 detected)...")
    rna_log = np.log2(rna_qc + 1)
else:
    print("Data appears already log-transformed — skipping log2 step.")
    rna_log = rna_qc.copy()

# Z-score per gene (mean=0, std=1)
scaler  = StandardScaler()
rna_arr = scaler.fit_transform(rna_log)
rna = pd.DataFrame(rna_arr, index=rna_log.index, columns=rna_log.columns)

print(f"  Final shape : {rna.shape}")
print(f"  Mean        : {rna.values.mean():.4f}  (expected ~0)")
print(f"  Std         : {rna.values.std():.4f}   (expected ~1)")

Data appears already log-transformed — skipping log2 step.
  Final shape : (1980, 20385)
  Mean        : 0.0000  (expected ~0)
  Std         : 1.0000   (expected ~1)


## 6. Variance pre-filter (bottom 25% removed)

In [13]:
variances     = rna.var(axis=0)
var_threshold = variances.quantile(0.25)
rna_var       = rna.loc[:, variances > var_threshold]

print(f"Variance pre-filter (remove bottom 25%)")
print(f"  Threshold   : {var_threshold:.4f}")
print(f"  Kept        : {rna_var.shape[1]:,} / {rna.shape[1]:,} genes")

# Variance distribution summary
v = variances[rna_var.columns]
print(f"\n  Variance stats (kept genes):")
print(f"    min    : {v.min():.4f}")
print(f"    median : {v.median():.4f}")
print(f"    mean   : {v.mean():.4f}")
print(f"    max    : {v.max():.4f}")

Variance pre-filter (remove bottom 25%)
  Threshold   : 1.0005
  Kept        : 14,435 / 20,385 genes

  Variance stats (kept genes):
    min    : 1.0005
    median : 1.0005
    mean   : 1.0005
    max    : 1.0005


## 7. Statistical scoring — Spearman + MI + Distance correlation

All scores are computed against `OS.time` (continuous survival time).
Results are cached to `rna_statistics_cache.csv` so this cell is fast on re-runs.

In [15]:
if STATS_CACHE.exists():
    print(f"Loading cached statistics from {STATS_CACHE.name}...")
    stats = pd.read_csv(STATS_CACHE)
    # Keep only genes still present in rna_var
    stats = stats[stats["gene"].isin(rna_var.columns)].reset_index(drop=True)
    print(f"  Loaded {len(stats):,} genes")

else:
    genes   = rna_var.columns.tolist()
    os_time = outcome["OS.time"].values
    n       = len(genes)

    print(f"Computing statistics for {n:,} genes...")
    print(f"  (This may take 15–30 min on first run; result is cached)")

    # --- [1/3] Spearman correlation + FDR ---
    print("  [1/3] Spearman correlation...")
    rho_arr  = np.zeros(n)
    pval_arr = np.zeros(n)
    for i, g in enumerate(genes):
        r, p       = spearmanr(rna_var[g].values, os_time)
        rho_arr[i]  = r if not np.isnan(r) else 0.0
        pval_arr[i] = p if not np.isnan(p) else 1.0
        if (i + 1) % 3000 == 0:
            print(f"    {i+1:,} / {n:,}")

    _, fdr_arr, _, _ = multipletests(pval_arr, method="fdr_bh")
    print(f"    FDR < 0.05: {(fdr_arr < 0.05).sum():,} genes")

    # --- [2/3] Mutual information ---
    print("  [2/3] Mutual information...")
    mi_arr = mutual_info_regression(
        rna_var.values, os_time,
        random_state=42, n_neighbors=5
    )

    # --- [3/3] Distance correlation (rank approximation) ---
    # Full O(n^2) distance correlation per gene is computationally prohibitive
    # at this scale. Pearson correlation on ranked values is a fast approximation
    # that captures non-linear monotone associations and adds ranking diversity.
    print("  [3/3] Distance correlation (rank approximation)...")
    os_rank = rankdata(os_time).astype(float)
    dc_arr  = []
    for g in genes:
        x_rank = rankdata(rna_var[g].values).astype(float)
        dc_arr.append(abs(float(np.corrcoef(x_rank, os_rank)[0, 1])))

    # Build stats DataFrame
    stats = pd.DataFrame({
        "gene":          genes,
        "spearman_rho":  rho_arr,
        "abs_spearman":  np.abs(rho_arr),
        "pval":          pval_arr,
        "fdr":           fdr_arr,
        "mutual_info":   mi_arr,
        "distance_corr": dc_arr,
        "variance":      variances[genes].values,
    })

    # Composite score: average rank across three methods (lower = better)
    stats["rank_spearman"] = rankdata(-stats["abs_spearman"])
    stats["rank_mi"]       = rankdata(-stats["mutual_info"])
    stats["rank_dcor"]     = rankdata(-stats["distance_corr"])
    stats["composite"]     = (
        stats["rank_spearman"] + stats["rank_mi"] + stats["rank_dcor"]
    ) / 3.0

    stats = stats.sort_values("composite").reset_index(drop=True)

    # Cache to disk
    STATS_CACHE.parent.mkdir(parents=True, exist_ok=True)
    stats.to_csv(STATS_CACHE, index=False)
    print(f"  Cached to {STATS_CACHE}")

# Summary
n_fdr = int((stats["fdr"] < 0.05).sum())
print(f"\nStatistics summary:")
print(f"  Total genes         : {len(stats):,}")
print(f"  FDR < 0.05          : {n_fdr:,}")
print(f"  |rho| range         : {stats['abs_spearman'].min():.4f} – {stats['abs_spearman'].max():.4f}")
print(f"  |rho| median        : {stats['abs_spearman'].median():.4f}")
print(f"\n  Top 10 by composite score:")
print(stats[["gene","abs_spearman","mutual_info","distance_corr","composite"]].head(10).to_string(index=False))

Computing statistics for 14,435 genes...
  (This may take 15–30 min on first run; result is cached)
  [1/3] Spearman correlation...
    3,000 / 14,435
    6,000 / 14,435
    9,000 / 14,435
    12,000 / 14,435
    FDR < 0.05: 4,993 genes
  [2/3] Mutual information...
  [3/3] Distance correlation (rank approximation)...
  Cached to C:\Users\olegk\Desktop\Thesis_v3\03_METABRIC_external_validation\v2\rna\rna_statistics_cache.csv

Statistics summary:
  Total genes         : 14,435
  FDR < 0.05          : 4,993
  |rho| range         : 0.0000 – 0.2150
  |rho| median        : 0.0369

  Top 10 by composite score:
  gene  abs_spearman  mutual_info  distance_corr  composite
GABPB1      0.197864     0.064232       0.197864   7.333333
  PGK1      0.195383     0.065966       0.195383   8.000000
 PREX1      0.205885     0.046775       0.205885  13.666667
  CD68      0.193546     0.049145       0.193546  16.000000
 PGAM1      0.179189     0.046635       0.179189  38.000000
 MED27      0.195937     0.0

## 8. Sanity checks before creating variants

In [17]:
# Confirm all stats genes are available in rna_var
missing_from_rna = set(stats["gene"]) - set(rna_var.columns)
print(f"Genes in stats but missing from rna_var : {len(missing_from_rna)}")

# Check outcome alignment
assert list(rna_var.index) == list(outcome.index), "Index mismatch between RNA and outcome!"
print(f"Index alignment OK: {len(rna_var)} samples")

# Variance percentile landmarks used by the 8 variants
for pct in [75, 95.9, 97.5, 98.3]:
    thresh = stats["variance"].quantile(pct / 100)
    count  = (stats["variance"] > thresh).sum()
    print(f"  Var > {pct:.1f}pct  ({thresh:.4f})  →  {count:,} genes")

# Correlation percentile landmarks
for pct in [97.5]:
    thresh = stats["abs_spearman"].quantile(pct / 100)
    count  = (stats["abs_spearman"] > thresh).sum()
    print(f"  |rho| > {pct:.1f}pct ({thresh:.4f})  →  {count:,} genes")

Genes in stats but missing from rna_var : 0
Index alignment OK: 1980 samples
  Var > 75.0pct  (1.0005)  →  3,545 genes
  Var > 95.9pct  (1.0005)  →  545 genes
  Var > 97.5pct  (1.0005)  →  342 genes
  Var > 98.3pct  (1.0005)  →  223 genes
  |rho| > 97.5pct (0.1364)  →  361 genes


## 9. Create 8 dataset variants

In [19]:
def build(gene_list, min_g=MIN_GENES):
    """Select genes from rna_var; pad with composite-ranked genes if below min_g."""
    gene_list = [g for g in gene_list if g in rna_var.columns]
    if len(gene_list) < min_g:
        have = set(gene_list)
        pad  = [g for g in stats["gene"] if g not in have]
        gene_list += pad[:min_g - len(gene_list)]
        print(f"    ⚠  Padded to {len(gene_list)} genes")
    # Preserve composite ordering (already sorted by composite in stats)
    order = {g: i for i, g in enumerate(stats["gene"])}
    gene_list = sorted(gene_list, key=lambda g: order.get(g, 9_999_999))
    return rna_var[gene_list]

print(f"Creating 8 dataset variants...")
print(f"Output: {OUTPUT_DIR}")
print()

created = []

# V1 — Ultra-conservative: variance > 98.3 percentile
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.983)]["gene"].tolist())
fname = f"rna_1_ultra_conservative_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V1", "name":"ultra_conservative", "n":len(df.columns),
                "logic":"Variance > 98.3 pct", "file":fname})
print(f"  V1 ultra_conservative : {len(df.columns):,} genes")

# V2 — Conservative: variance > 97.5 percentile
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.975)]["gene"].tolist())
fname = f"rna_2_conservative_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V2", "name":"conservative", "n":len(df.columns),
                "logic":"Variance > 97.5 pct", "file":fname})
print(f"  V2 conservative       : {len(df.columns):,} genes")

# V3 — Standard: variance > 95.9 percentile
df    = build(stats[stats["variance"] > stats["variance"].quantile(0.959)]["gene"].tolist())
fname = f"rna_3_standard_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V3", "name":"standard", "n":len(df.columns),
                "logic":"Variance > 95.9 pct", "file":fname})
print(f"  V3 standard           : {len(df.columns):,} genes")

# V4 — FDR-significant: Spearman FDR < 0.05, top TARGET_BROAD by composite
fdr_genes = stats[stats["fdr"] < 0.05].head(TARGET_BROAD)["gene"].tolist()
df    = build(fdr_genes, min_g=TARGET_BROAD)
fname = f"rna_4_fdr_significant_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V4", "name":"fdr_significant", "n":len(df.columns),
                "logic":"FDR<0.05, top 1000 by composite", "file":fname})
print(f"  V4 fdr_significant    : {len(df.columns):,} genes  "
      f"(raw FDR<0.05: {int((stats['fdr']<0.05).sum())})")

# V5 — Balanced: top 25% variance then top 10% Spearman within that subset
var_sub     = stats[stats["variance"] > stats["variance"].quantile(0.75)]
corr_thresh = var_sub["abs_spearman"].quantile(0.90)
df    = build(var_sub[var_sub["abs_spearman"] > corr_thresh]["gene"].tolist())
fname = f"rna_5_balanced_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V5", "name":"balanced", "n":len(df.columns),
                "logic":"Var top 25% → Spearman top 10%", "file":fname})
print(f"  V5 balanced           : {len(df.columns):,} genes")

# V6 — Correlation-based: abs Spearman > 97.5 percentile
df    = build(stats[stats["abs_spearman"] > stats["abs_spearman"].quantile(0.975)]["gene"].tolist())
fname = f"rna_6_correlation_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V6", "name":"correlation", "n":len(df.columns),
                "logic":"Abs Spearman > 97.5 pct", "file":fname})
print(f"  V6 correlation        : {len(df.columns):,} genes")

# V7 — Top correlated: top 100 positive + top 100 negative Spearman rho
by_rho  = stats.sort_values("spearman_rho", ascending=False)
top_pos = by_rho[by_rho["spearman_rho"] > 0].head(100)["gene"].tolist()
top_neg = by_rho[by_rho["spearman_rho"] < 0].tail(100)["gene"].tolist()
df      = build(sorted(set(top_pos + top_neg)))
# Save annotated version with scores for inspection
stats[stats["gene"].isin(set(top_pos + top_neg))][
    ["gene", "spearman_rho", "pval", "fdr", "mutual_info", "variance"]
].to_csv(OUTPUT_DIR / "rna_7_top_correlated_annotated.csv", index=False)
fname = f"rna_7_top_correlated_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V7", "name":"top_correlated", "n":len(df.columns),
                "logic":"Top 100 pos + top 100 neg Spearman", "file":fname})
print(f"  V7 top_correlated     : {len(df.columns):,} genes")

# V8 — Composite: top TARGET_BROAD by average rank of Spearman + MI + Dcor
df    = build(stats.head(TARGET_BROAD)["gene"].tolist(), min_g=TARGET_BROAD)
fname = f"rna_8_composite_{len(df.columns)}genes.csv"
df.to_csv(OUTPUT_DIR / fname)
created.append({"V":"V8", "name":"composite", "n":len(df.columns),
                "logic":"Avg rank Spearman+MI+Dcor, top 1000", "file":fname})
print(f"  V8 composite          : {len(df.columns):,} genes")

# Auxiliary files
outcome.to_csv(OUTPUT_DIR / "outcome.csv")
stats.to_csv(OUTPUT_DIR / "rna_statistics_complete.csv", index=False)
summary = pd.DataFrame(created)
summary.to_csv(OUTPUT_DIR / "datasets_summary.csv", index=False)

print()
print(summary[["V", "name", "n", "logic"]].to_string(index=False))
print(f"\nSaved to: {OUTPUT_DIR}")

Creating 8 dataset variants...
Output: C:\Users\olegk\Desktop\Thesis_v3\03_METABRIC_external_validation\v2\rna\statistical_filtered

  V1 ultra_conservative : 223 genes
  V2 conservative       : 342 genes
  V3 standard           : 545 genes
  V4 fdr_significant    : 1,000 genes  (raw FDR<0.05: 4993)
  V5 balanced           : 355 genes
  V6 correlation        : 361 genes
  V7 top_correlated     : 200 genes
  V8 composite          : 1,000 genes

 V               name    n                               logic
V1 ultra_conservative  223                 Variance > 98.3 pct
V2       conservative  342                 Variance > 97.5 pct
V3           standard  545                 Variance > 95.9 pct
V4    fdr_significant 1000     FDR<0.05, top 1000 by composite
V5           balanced  355      Var top 25% → Spearman top 10%
V6        correlation  361             Abs Spearman > 97.5 pct
V7     top_correlated  200  Top 100 pos + top 100 neg Spearman
V8          composite 1000 Avg rank Spearman+MI+

## 10. Final verification

In [21]:
print("Verifying saved files...")
print()

dataset_files = sorted(OUTPUT_DIR.glob("rna_*genes.csv"))
print(f"  Found {len(dataset_files)} dataset files:")

ok = True
for fpath in dataset_files:
    df_check = pd.read_csv(fpath, index_col=0)
    n_samples, n_genes = df_check.shape
    has_nan   = df_check.isna().any().any()
    has_inf   = np.isinf(df_check.values).any()
    status    = "OK" if not has_nan and not has_inf else "⚠ NaN/Inf!"
    if has_nan or has_inf:
        ok = False
    print(f"    {fpath.name:<55} {n_samples} samples × {n_genes:>5} genes  [{status}]")

other_files = ["outcome.csv", "rna_statistics_complete.csv", "datasets_summary.csv",
               "rna_7_top_correlated_annotated.csv"]
print()
for fname in other_files:
    p = OUTPUT_DIR / fname
    print(f"    {'✓' if p.exists() else '✗'} {fname}")

print()
if ok:
    print("All datasets passed — no NaN or Inf values found.")
    print(f"Next step: run  03_run_MB_rna.py  (point FILTERED_DIR to {OUTPUT_DIR})")
else:
    print("⚠  Some datasets contain NaN or Inf — investigate before running MB!")

Verifying saved files...

  Found 8 dataset files:
    rna_1_ultra_conservative_223genes.csv                   1980 samples ×   223 genes  [OK]
    rna_2_conservative_342genes.csv                         1980 samples ×   342 genes  [OK]
    rna_3_standard_545genes.csv                             1980 samples ×   545 genes  [OK]
    rna_4_fdr_significant_1000genes.csv                     1980 samples ×  1000 genes  [OK]
    rna_5_balanced_355genes.csv                             1980 samples ×   355 genes  [OK]
    rna_6_correlation_361genes.csv                          1980 samples ×   361 genes  [OK]
    rna_7_top_correlated_200genes.csv                       1980 samples ×   200 genes  [OK]
    rna_8_composite_1000genes.csv                           1980 samples ×  1000 genes  [OK]

    ✓ outcome.csv
    ✓ rna_statistics_complete.csv
    ✓ datasets_summary.csv
    ✓ rna_7_top_correlated_annotated.csv

All datasets passed — no NaN or Inf values found.
Next step: run  03_run_MB_rna.py 